In [ ]:
# Cài đặt pyodbc nếu chưa có
!pip install pyodbc

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Cell 1: Imports và thiết lập thiết bị
import os
import pyodbc
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import json
import sys

# Thiết lập thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sử dụng thiết bị: {device}")

# Tạo thư mục models nếu chưa có
os.makedirs('models', exist_ok=True)


Sử dụng thiết bị: cpu


In [ ]:
# Cell 2: Cấu hình tập trung cho Google Colab (thay đổi ở đây cho cả 3 models)
# Phát hiện GPU và đề xuất batch_size
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f"Device: {device}, GPU: {gpu_name}")

# === CẤU HÌNH ADAM OPTIMIZER ===
OPTIMIZER_NAME = 'Adam'  # Tên optimizer để phân biệt

# Hyperparameters chung cho cả 3 models
BATCH_SIZE = 32          # Colab GPU T4/P100: 16-32, V100/A100: 32-64
NUM_EPOCHS = 20          # Transfer learning (freeze backbone): 10-20
PATIENCE = 5             # Early stopping patience

# Adam Learning rates
ADAM_LR = 1e-3           # ResNet50, MobileNetV2
ADAM_LR_EFFICIENTNET = 5e-4  # EfficientNet-B0 (nhạy cảm hơn)
ADAM_WEIGHT_DECAY = 1e-4     # Weight decay cho Adam (L2 regularization)

USE_AMP = torch.cuda.is_available()  # Bật mixed precision nếu có GPU

# Đường dẫn Google Drive (riêng cho Adam - plant_type)
DRIVE_BASE = '/content/drive/MyDrive/plant_type_adam'
MODELS_DIR = f'{DRIVE_BASE}/models'
PLOTS_DIR = f'{DRIVE_BASE}/plots'
CHECKPOINTS_DIR = f'{DRIVE_BASE}/checkpoints'

print(f"\n=== CẤU HÌNH ADAM - PLANT TYPE ===")
print(f"Optimizer: {OPTIMIZER_NAME}")
print(f"Task: Plant Type Classification (9 classes)")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Num Epochs: {NUM_EPOCHS}")
print(f"Patience: {PATIENCE}")
print(f"Adam Learning Rate: {ADAM_LR}")
print(f"Adam EfficientNet LR: {ADAM_LR_EFFICIENTNET}")
print(f"Adam Weight Decay: {ADAM_WEIGHT_DECAY}")
print(f"Use AMP: {USE_AMP}")
print(f"Base Dir: {DRIVE_BASE}")


In [ ]:
# Cell 3: Giải nén file ZIP và tạo cấu trúc thư mục
import zipfile
import time
import sys

# Dùng chung file ZIP từ thư mục data
zip_path = '/content/drive/MyDrive/data/plant_images.zip'
extract_path = '/content/'

os.makedirs(extract_path, exist_ok=True)
print("Đang giải nén file ZIP...")
start_time = time.time()

if not os.path.exists(zip_path):
    print(f"Lỗi: Không tìm thấy file ZIP tại {zip_path}. Vui lòng kiểm tra đường dẫn.")
    sys.exit()
else:
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        extract_time = time.time() - start_time
        print(f"Thời gian giải nén: {extract_time:.2f} giây")

        def count_files_in_directory(directory):
            total_files = 0
            for root, dirs, files in os.walk(directory):
                total_files += len(files)
            return total_files

        plant_images_dir = os.path.join(extract_path, 'plant_images')
        if os.path.exists(plant_images_dir):
            total_files = count_files_in_directory(plant_images_dir)
            print(f"Tổng số file đã giải nén trong thư mục plant_images: {total_files}")
        else:
            print("Thư mục plant_images không tồn tại sau khi giải nén!")
            sys.exit()

        print("Cấu trúc thư mục sau khi giải nén:")
        for root, dirs, files in os.walk(plant_images_dir):
            level = root.replace(extract_path, '').count('/')
            indent = ' ' * 4 * level
            print(f"{indent}{os.path.basename(root)}/")
            for file in files[:3]:
                print(f"{indent}    {file}")
    except zipfile.BadZipFile:
        print(f"Lỗi: File zip tại {zip_path} bị hỏng.")
        sys.exit()
    except Exception as e:
        print(f"Đã xảy ra lỗi khi giải nén: {e}")
        sys.exit()

# Tạo thư mục models cho ResNet50, MobileNetV2, EfficientNet-B0 (dùng biến cấu hình)
for model_type in ['resnet50', 'mobilenetv2', 'efficientnetb0']:
    os.makedirs(f'{MODELS_DIR}/{model_type}/best', exist_ok=True)
    os.makedirs(f'{MODELS_DIR}/{model_type}/final', exist_ok=True)
    os.makedirs(f'{CHECKPOINTS_DIR}/{model_type}', exist_ok=True)

os.makedirs(PLOTS_DIR, exist_ok=True)
print("\n✓ Đã tạo cấu trúc thư mục cho 3 models")

In [ ]:
# Cell 4: Đọc dữ liệu từ CSV (dùng CSV chung từ thư mục data)
media_root = '/content/'  # Thư mục chứa ảnh sau giải nén
csv_path = '/content/drive/MyDrive/data/plant_data.csv'  # CSV chung cho cả 2 notebooks

try:
    df = pd.read_csv(csv_path)
    print("Đã tải dữ liệu từ CSV")
    print(f"Tổng số ảnh: {len(df)}")
except Exception as e:
    print(f"Lỗi khi đọc CSV: {e}")
    sys.exit()

C:\Users\DELL\AppData\Local\Temp\ipykernel_11700\308630144.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Đã tải dữ liệu từ SQL Server
Tổng số ảnh: 87867


In [ ]:
# Block 3: Chuẩn hóa dữ liệu và ánh xạ plant_type (loại bỏ cây chỉ có 1 class)
if df.empty:
    print("Lỗi: DataFrame rỗng. Kiểm tra dữ liệu trong bảng plant_health_app_plantmodel.")
    sys.exit()

df['image'] = df['image'].apply(lambda x: x.replace('\\', '/').strip())

# In các giá trị plant_type gốc để debug
print("Plant types trước ánh xạ:", df['plant_type'].unique())

# Ánh xạ plant_type - loại bỏ Blueberry, Orange, Raspberry, Soybean, Squash (chỉ có 1 class)
PLANT_TYPE_MAPPING = {
    "Cherry": "Cherry",
    "Corn": "Corn",
    "Pepper": "Pepper",
    "Apple": "Apple",
    "Grape": "Grape",
    "Peach": "Peach",
    "Potato": "Potato",
    "Strawberry": "Strawberry",
    "Tomato": "Tomato",
    # Loại bỏ: "Blueberry", "Orange", "Raspberry", "Soybean", "Squash"
}
df['plant_type'] = df['plant_type'].map(PLANT_TYPE_MAPPING).fillna('Unknown')

# Loại bỏ các hàng có plant_type là 'Unknown' (Blueberry, Orange, Raspberry, Soybean, Squash)
unknown_count = df['plant_type'].eq('Unknown').sum()
if unknown_count > 0:
    print(f"Loại bỏ {unknown_count} hàng có plant_type không hợp lệ (Blueberry, Orange, Raspberry, Soybean, Squash).")
    df = df[df['plant_type'] != 'Unknown']

# Tách dữ liệu thành train và validation
train_df = df[df['dataset_type'] == 'train']
val_df = df[df['dataset_type'] == 'valid']
print(f"Số ảnh train: {len(train_df)}")
print(f"Số ảnh validation: {len(val_df)}")

if train_df.empty or val_df.empty:
    print("Lỗi: Tập train hoặc validation rỗng. Kiểm tra cột dataset_type trong SQL.")
    sys.exit()

# Kiểm tra dữ liệu sau ánh xạ
print("Plant types sau ánh xạ:", df['plant_type'].unique())

Plant types trước ánh xạ: ['Apple' 'Blueberry' 'Cherry' 'Corn' 'Grape' 'Orange' 'Peach' 'Pepper'
 'Potato' 'Raspberry' 'Soybean' 'Squash' 'Strawberry' 'Tomato']
Số ảnh train: 70295
Số ảnh validation: 17572
Plant types sau ánh xạ: ['Apple' 'Blueberry' 'Cherry' 'Corn' 'Grape' 'Orange' 'Peach' 'Pepper'
 'Potato' 'Raspberry' 'Soybean' 'Squash' 'Strawberry' 'Tomato']


In [ ]:
# Cell 6: Định nghĩa PLANT_CLASSES và lưu JSON (đã loại bỏ cây chỉ có 1 class)
# Loại bỏ: Blueberry, Orange, Raspberry, Soybean, Squash (chỉ có 1 class không train được)
PLANT_CLASSES = [
    "Apple", "Cherry", "Corn", "Grape", "Peach", 
    "Pepper", "Potato", "Strawberry", "Tomato"
]
num_classes = len(PLANT_CLASSES)
print(f"Số lớp cây: {num_classes}")
print(f"Danh sách lớp cây: {PLANT_CLASSES}")

# Lưu plant_classes.json
with open('plant_classes.json', 'w', encoding='utf-8') as f:
    json.dump(PLANT_CLASSES, f, ensure_ascii=False)
print("Đã lưu plant_classes.json")

Số lớp cây: 14
Danh sách lớp cây: ['Apple', 'Blueberry', 'Cherry', 'Corn', 'Grape', 'Orange', 'Peach', 'Pepper', 'Potato', 'Raspberry', 'Soybean', 'Squash', 'Strawberry', 'Tomato']
Đã lưu plant_classes.json


In [11]:
# Cell 5: Tạo danh sách ảnh và nhãn
train_image_paths = [os.path.join(media_root, row['image']) for _, row in train_df.iterrows()]
train_labels = [PLANT_CLASSES.index(row['plant_type']) for _, row in train_df.iterrows()]

val_image_paths = [os.path.join(media_root, row['image']) for _, row in val_df.iterrows()]
val_labels = [PLANT_CLASSES.index(row['plant_type']) for _, row in val_df.iterrows()]

print(f"Số lượng ảnh train: {len(train_image_paths)}")
print(f"Số lượng ảnh validation: {len(val_image_paths)}")

Số lượng ảnh train: 70295
Số lượng ảnh validation: 17572


In [12]:
# Cell 6: Định nghĩa PlantDiseaseDataset class
class PlantDiseaseDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.valid_indices = []
        
        print(f"Đang kiểm tra {len(image_paths)} ảnh...")
        start_time = time.time()
        for idx, img_path in tqdm(enumerate(image_paths), total=len(image_paths), desc="Kiểm tra ảnh"):
            try:
                with Image.open(img_path) as img:
                    img.verify()
                self.valid_indices.append(idx)
            except Exception as e:
                print(f"Lỗi khi mở ảnh {img_path}: {e}")
        print(f"Thời gian kiểm tra ảnh: {time.time() - start_time:.2f} giây")
        print(f"Số ảnh hợp lệ: {len(self.valid_indices)}/{len(image_paths)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        true_idx = self.valid_indices[idx]
        img_path = self.image_paths[true_idx]
        label = self.labels[true_idx]
        
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Lỗi khi mở ảnh {img_path}: {e}")
            raise RuntimeError(f"Không thể mở ảnh {img_path}.")

        if self.transform:
            try:
                image = self.transform(image)
            except Exception as e:
                print(f"Lỗi khi biến đổi ảnh {img_path}: {e}")
                raise RuntimeError(f"Không thể biến đổi ảnh {img_path}.")

        return image, label

In [ ]:
# Block 7: Tạo dataset và DataLoader (dùng BATCH_SIZE từ cấu hình)
# ResNet18 và MobileNetV2: 224x224
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# EfficientNet-B0: 224x224 trực tiếp (không qua resize 256)
efficientnet_transforms = {
    'train': transforms.Compose([
        transforms.Resize(224),  # Resize trực tiếp về 224
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),  # Thêm augmentation
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),  # Resize trực tiếp về 224
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

start_dataset_time = time.time()
train_dataset = PlantDiseaseDataset(train_image_paths, train_labels, transform=data_transforms['train'])
val_dataset = PlantDiseaseDataset(val_image_paths, val_labels, transform=data_transforms['val'])
train_dataset = Subset(train_dataset, train_dataset.valid_indices)
val_dataset = Subset(val_dataset, val_dataset.valid_indices)
print(f"Thời gian tạo dataset: {time.time() - start_dataset_time:.2f} giây")

start_loader_time = time.time()
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, drop_last=False)
print(f"Thời gian tạo DataLoader: {time.time() - start_loader_time:.2f} giây")
print(f"Số batch train: {len(train_loader)}")
print(f"Số batch validation: {len(val_loader)}")
print(f"Batch size được sử dụng: {BATCH_SIZE}")


Đang kiểm tra 70295 ảnh...


Kiểm tra ảnh:   0%|          | 0/70295 [00:00<?, ?it/s]

Kiểm tra ảnh: 100%|██████████| 70295/70295 [16:31<00:00, 70.92it/s]  


Thời gian kiểm tra ảnh: 991.26 giây
Số ảnh hợp lệ: 70295/70295
Đang kiểm tra 17572 ảnh...


Kiểm tra ảnh: 100%|██████████| 17572/17572 [04:06<00:00, 71.32it/s]

Thời gian kiểm tra ảnh: 246.38 giây
Số ảnh hợp lệ: 17572/17572
Thời gian tạo dataset: 1237.64 giây
Thời gian tạo DataLoader: 0.01 giây
Số batch train: 8786
Số batch validation: 2196


# Định nghĩa hàm huấn luyện chung

Hàm `train_model()` dùng chung cho cả 3 models (ResNet18, MobileNetV2, EfficientNet-B0)

In [ ]:
# Cell: Hàm lưu và tải checkpoint (resume training nếu bị disconnect)
def save_checkpoint(model, optimizer, epoch, train_losses, val_losses, train_accs, val_accs, checkpoint_path, model_name):
    """
    Lưu checkpoint sau mỗi epoch để resume training nếu Colab disconnect.
    
    Args:
        model: PyTorch model
        optimizer: Optimizer đang sử dụng
        epoch: Epoch hiện tại (0-indexed)
        train_losses, val_losses, train_accs, val_accs: Danh sách metrics
        checkpoint_path: Đường dẫn lưu checkpoint
        model_name: Tên model (resnet18, mobilenetv2, efficientnetb0)
    """
    checkpoint = {
        'epoch': epoch + 1,  # Lưu epoch tiếp theo để resume
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accs': train_accs,
        'val_accs': val_accs,
        'model_name': model_name
    }
    try:
        torch.save(checkpoint, checkpoint_path)
        # Không in thông báo để giảm clutter
    except Exception as e:
        print(f"⚠️ Lỗi khi lưu checkpoint: {e}")

def load_checkpoint(model, optimizer, checkpoint_path, device):
    """
    Tải checkpoint nếu tồn tại để resume training.
    
    Returns:
        tuple: (start_epoch, train_losses, val_losses, train_accs, val_accs)
               - start_epoch: Epoch bắt đầu (nếu không có checkpoint thì 0)
               - Các list metrics từ checkpoint (nếu không có thì list rỗng)
    """
    if os.path.exists(checkpoint_path):
        try:
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            epoch = checkpoint['epoch']
            train_losses = checkpoint['train_losses']
            val_losses = checkpoint['val_losses']
            train_accs = checkpoint['train_accs']
            val_accs = checkpoint['val_accs']
            print(f"✓ Đã tải checkpoint từ epoch {epoch} (resume training)")
            return epoch, train_losses, val_losses, train_accs, val_accs
        except Exception as e:
            print(f"⚠️ Lỗi khi tải checkpoint: {e}. Huấn luyện từ đầu.")
            return 0, [], [], [], []
    else:
        print("Không tìm thấy checkpoint. Huấn luyện từ đầu.")
        return 0, [], [], [], []


In [ ]:
# Cell: Hàm train_model với checkpoint support (giữ progress bar, bỏ thông báo dài)
def train_model(model, criterion, optimizer, train_loader, val_loader, num_epochs, patience, device, 
                model_path, best_model_path, checkpoint_path, model_name, verbose=1):
    """
    Huấn luyện model với checkpoint support và progress bar.
    
    Checkpoint lưu progress sau mỗi epoch để resume nếu bị gián đoạn (Colab timeout, crash).
    """
    model = model.to(device)
    
    # Tải checkpoint nếu có (resume training)
    start_epoch, train_losses, val_losses, train_accs, val_accs = load_checkpoint(model, optimizer, checkpoint_path, device)
    best_val_acc = float('-inf') if not val_accs else max(val_accs)
    epochs_no_improve = 0
    start_time = time.time()

    for epoch in range(start_epoch, num_epochs):
        # Train
        model.train()
        running_loss, running_corrects, total_samples = 0.0, 0, 0
        
        for inputs, labels in tqdm(train_loader, desc=f"Huấn luyện (Epoch {epoch+1})", ncols=100, file=sys.stdout):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)
        
        train_loss = running_loss / total_samples
        train_acc = (running_corrects.double() / total_samples).item()
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        print(f"Huấn luyện - Loss: {train_loss:.6f} Acc: {train_acc:.4f}")
        
        # Validation
        model.eval()
        running_loss, running_corrects, total_samples = 0.0, 0, 0
        
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Kiểm tra (Epoch {epoch+1})", ncols=100, file=sys.stdout):
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                running_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                running_corrects += torch.sum(preds == labels.data)
                total_samples += inputs.size(0)
        
        val_loss = running_loss / total_samples
        val_acc = (running_corrects.double() / total_samples).item()
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        print(f"Kiểm tra - Loss: {val_loss:.6f} Acc: {val_acc:.4f}")
        
        # Save checkpoint (silent)
        save_checkpoint(model, optimizer, epoch, train_losses, val_losses, train_accs, val_accs, checkpoint_path, model_name)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"✓ Best: {best_val_acc:.4f}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"⏹ Early stop")
                break
    
    # Save final model
    torch.save(model.state_dict(), model_path)
    
    # Save plot (silent)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train', color='blue')
    plt.plot(val_losses, label='Val', color='orange')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title(f'Loss - {model_name}')
    
    plt.subplot(1, 2, 2)
    plt.plot(train_accs, label='Train', color='blue')
    plt.plot(val_accs, label='Val', color='orange')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title(f'Accuracy - {model_name}')
    plt.tight_layout()
    
    model_plots_dir = f'{PLOTS_DIR}/{model_name}'
    os.makedirs(model_plots_dir, exist_ok=True)
    plot_path = f'{model_plots_dir}/plant_type_training_plot.png'
    plt.savefig(plot_path)
    plt.close()
    
    print(f"\n✓ {model_name} hoàn tất ({(time.time() - start_time)/60:.1f} phút)\n")
    
    return model, {'train_losses': train_losses, 'val_losses': val_losses, 'train_accs': train_accs, 'val_accs': val_accs}


# Huấn luyện 3 models để so sánh

Cả 3 models đều sử dụng transfer learning (freeze backbone, chỉ train classifier) với cùng cấu hình từ Cell 2.
- **ResNet50**: IMAGENET1K_V2 weights (80.9% ImageNet accuracy)
- **MobileNetV2**: IMAGENET1K_V1 weights (71.9% ImageNet accuracy)
- **EfficientNet-B0**: IMAGENET1K_V1 weights (77.7% ImageNet accuracy)

In [ ]:
# Cell 9: Huấn luyện ResNet50 (dùng cấu hình tập trung + checkpoint)
print("\n=== Huấn luyện ResNet50 ===")

# Khởi tạo model ResNet50 (V2 weights - tốt hơn V1)
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Đóng băng tất cả các layer trừ classifier
for param in model.parameters():
    param.requires_grad = False

# Thay classifier cuối
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# Adam optimizer với weight decay
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=ADAM_LR, weight_decay=ADAM_WEIGHT_DECAY)
print(f"ResNet50 + Adam (LR={ADAM_LR}, Weight Decay={ADAM_WEIGHT_DECAY})")

# Đường dẫn lưu model (Google Drive - dùng biến từ cấu hình)
resnet_model_path = f'{MODELS_DIR}/resnet50/final/resnet50_plant_type_model.pth'
resnet_best_model_path = f'{MODELS_DIR}/resnet50/best/best_resnet50_plant_type_model.pth'
resnet_checkpoint_path = f'{CHECKPOINTS_DIR}/resnet50/resnet50_checkpoint.pth'

print(f"ResNet50 đã được khởi tạo với {num_classes} lớp (IMAGENET1K_V2 weights)")

# Huấn luyện (dùng NUM_EPOCHS và PATIENCE từ cấu hình, truyền checkpoint_path)
model, history = train_model(
    model, criterion, optimizer, 
    train_loader, val_loader, 
    num_epochs=NUM_EPOCHS, 
    patience=PATIENCE, 
    device=device, 
    model_path=resnet_model_path, 
    best_model_path=resnet_best_model_path,
    checkpoint_path=resnet_checkpoint_path,
    model_name='resnet50',
    verbose=1
)

print(f"✓ ResNet50 hoàn tất: {resnet_model_path}")


In [ ]:
# Cell 10: Huấn luyện MobileNetV2 (dùng cấu hình tập trung + checkpoint)
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

print("\n=== Huấn luyện MobileNetV2 ===")
mobilenet_model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)

# Đóng băng tất cả các layer trừ classifier
for param in mobilenet_model.parameters():
    param.requires_grad = False

# Thay classifier cuối
mobilenet_model.classifier[1] = nn.Linear(mobilenet_model.classifier[1].in_features, num_classes)
mobilenet_model = mobilenet_model.to(device)

# Adam optimizer với weight decay
mobilenet_criterion = nn.CrossEntropyLoss()
mobilenet_optimizer = optim.Adam(mobilenet_model.classifier.parameters(), lr=ADAM_LR, weight_decay=ADAM_WEIGHT_DECAY)
print(f"MobileNetV2 + Adam (LR={ADAM_LR}, Weight Decay={ADAM_WEIGHT_DECAY})")

# Đường dẫn lưu model (Google Drive - dùng biến từ cấu hình)
mobilenet_model_path = f'{MODELS_DIR}/mobilenetv2/final/mobilenetv2_plant_type_model.pth'
mobilenet_best_model_path = f'{MODELS_DIR}/mobilenetv2/best/best_mobilenetv2_plant_type_model.pth'
mobilenet_checkpoint_path = f'{CHECKPOINTS_DIR}/mobilenetv2/mobilenetv2_checkpoint.pth'

# Huấn luyện (dùng NUM_EPOCHS và PATIENCE từ cấu hình, truyền checkpoint_path)
mobilenet_model, mobilenet_history = train_model(
    mobilenet_model, mobilenet_criterion, mobilenet_optimizer, 
    train_loader, val_loader, 
    num_epochs=NUM_EPOCHS, 
    patience=PATIENCE, 
    device=device,
    model_path=mobilenet_model_path, 
    best_model_path=mobilenet_best_model_path,
    checkpoint_path=mobilenet_checkpoint_path,
    model_name='mobilenetv2',
    verbose=1
)

print(f"✓ MobileNetV2 hoàn tất: {mobilenet_model_path}")


# Chuẩn bị Dataset cho EfficientNet-B0

EfficientNet-B0 yêu cầu transforms riêng với resize trực tiếp về 224x224 (không qua 256)

In [ ]:
# Cell 11: Tạo DataLoader riêng cho EfficientNet-B0 (dùng efficientnet_transforms)
print("\n=== Chuẩn bị Dataset cho EfficientNet-B0 ===")

start_dataset_time = time.time()
efficientnet_train_dataset = PlantDiseaseDataset(train_image_paths, train_labels, transform=efficientnet_transforms['train'])
efficientnet_val_dataset = PlantDiseaseDataset(val_image_paths, val_labels, transform=efficientnet_transforms['val'])
efficientnet_train_dataset = Subset(efficientnet_train_dataset, efficientnet_train_dataset.valid_indices)
efficientnet_val_dataset = Subset(efficientnet_val_dataset, efficientnet_val_dataset.valid_indices)
print(f"Thời gian tạo dataset cho EfficientNet: {time.time() - start_dataset_time:.2f} giây")

start_loader_time = time.time()
efficientnet_train_loader = DataLoader(efficientnet_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
efficientnet_val_loader = DataLoader(efficientnet_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, drop_last=False)
print(f"Thời gian tạo DataLoader cho EfficientNet: {time.time() - start_loader_time:.2f} giây")
print(f"Số batch train: {len(efficientnet_train_loader)}")
print(f"Số batch validation: {len(efficientnet_val_loader)}")


In [ ]:
# Cell 12: Huấn luyện EfficientNet-B0 (dùng transforms + LR riêng + DataLoader riêng)
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

print("\n=== Huấn luyện EfficientNet-B0 ===")
efficientnet_model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

# Đóng băng tất cả các layer trừ classifier
for param in efficientnet_model.parameters():
    param.requires_grad = False

# Thay classifier cuối
efficientnet_model.classifier[1] = nn.Linear(efficientnet_model.classifier[1].in_features, num_classes)
efficientnet_model = efficientnet_model.to(device)

# Adam optimizer với LR thấp hơn và weight decay
efficientnet_criterion = nn.CrossEntropyLoss()
efficientnet_optimizer = optim.Adam(efficientnet_model.classifier.parameters(), lr=ADAM_LR_EFFICIENTNET, weight_decay=ADAM_WEIGHT_DECAY)

# Đường dẫn lưu model (Google Drive - dùng biến từ cấu hình)
efficientnet_model_path = f'{MODELS_DIR}/efficientnetb0/final/efficientnetb0_plant_type_model.pth'
efficientnet_best_model_path = f'{MODELS_DIR}/efficientnetb0/best/best_efficientnetb0_plant_type_model.pth'
efficientnet_checkpoint_path = f'{CHECKPOINTS_DIR}/efficientnetb0/efficientnetb0_checkpoint.pth'

print(f"EfficientNet-B0 + Adam (LR={ADAM_LR_EFFICIENTNET}, Weight Decay={ADAM_WEIGHT_DECAY})")

# Huấn luyện (dùng efficientnet_train_loader và efficientnet_val_loader riêng)
efficientnet_model, efficientnet_history = train_model(
    efficientnet_model, efficientnet_criterion, efficientnet_optimizer,
    efficientnet_train_loader, efficientnet_val_loader,  # DataLoader riêng cho EfficientNet
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    device=device,
    model_path=efficientnet_model_path,
    best_model_path=efficientnet_best_model_path,
    checkpoint_path=efficientnet_checkpoint_path,
    model_name='efficientnetb0',
    verbose=1
)

print(f"✓ EfficientNet-B0 hoàn tất: {efficientnet_model_path}")


In [ ]:
# Cell 13: So sánh kết quả 3 models
import pandas as pd

print("\n=== SO SÁNH KẾT QUẢ 3 MODELS ===\n")

# Tạo DataFrame so sánh
comparison_data = {
    'Model': ['ResNet50', 'MobileNetV2', 'EfficientNet-B0'],
    'Best Val Accuracy': [
        max(history['val_accs']) if history['val_accs'] else 0,
        max(mobilenet_history['val_accs']) if mobilenet_history['val_accs'] else 0,
        max(efficientnet_history['val_accs']) if efficientnet_history['val_accs'] else 0
    ],
    'Final Train Loss': [
        history['train_losses'][-1] if history['train_losses'] else 0,
        mobilenet_history['train_losses'][-1] if mobilenet_history['train_losses'] else 0,
        efficientnet_history['train_losses'][-1] if efficientnet_history['train_losses'] else 0
    ],
    'Final Val Loss': [
        history['val_losses'][-1] if history['val_losses'] else 0,
        mobilenet_history['val_losses'][-1] if mobilenet_history['val_losses'] else 0,
        efficientnet_history['val_losses'][-1] if efficientnet_history['val_losses'] else 0
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

# Vẽ biểu đồ so sánh
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# So sánh Validation Accuracy
axes[0].plot(history['val_accs'], label='ResNet50', marker='o')
axes[0].plot(mobilenet_history['val_accs'], label='MobileNetV2', marker='s')
axes[0].plot(efficientnet_history['val_accs'], label='EfficientNet-B0', marker='^')
axes[0].set_title('Validation Accuracy Comparison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# So sánh Validation Loss
axes[1].plot(history['val_losses'], label='ResNet50', marker='o')
axes[1].plot(mobilenet_history['val_losses'], label='MobileNetV2', marker='s')
axes[1].plot(efficientnet_history['val_losses'], label='EfficientNet-B0', marker='^')
axes[1].set_title('Validation Loss Comparison')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()

# Lưu vào thư mục comparison (dùng biến PLOTS_DIR từ cấu hình)
comparison_dir = f'{PLOTS_DIR}/comparison'
os.makedirs(comparison_dir, exist_ok=True)
comparison_plot_path = f'{comparison_dir}/plant_type_model_comparison_adam.png'
plt.savefig(comparison_plot_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Biểu đồ so sánh đã lưu tại: {comparison_plot_path}")
print(f"\n=== TỔNG KẾT ADAM - PLANT TYPE ===")
print(f"Task: Plant Type Classification (9 classes)")
print(f"Optimizer: Adam (L2 weight decay)")
print(f"Cấu hình: Batch={BATCH_SIZE}, Epochs={NUM_EPOCHS}, Patience={PATIENCE}")
print(f"LR: ResNet50/MobileNetV2={ADAM_LR}, EfficientNet-B0={ADAM_LR_EFFICIENTNET}")
print(f"Weight Decay: {ADAM_WEIGHT_DECAY}")
print(f"3 models: ResNet50, MobileNetV2, EfficientNet-B0")
print(f"Models lưu tại: {MODELS_DIR}")
